In [6]:
import pymysql
import pandas as pd
import decimal

# Database Connection Details
DB_CONFIG = {
    'host': 'localhost',
    'user': 'towdevuser',
    'password': 'Dev703',
    'database': 'timelineofwealth'
}

# Define the ticker as a constant
TICKER = 'VENUSPIPES'  # Change this to analyze other tickers

# Queries for fetching data
QUERIES = {
    'stock_quarter': f"SELECT * FROM stock_quarter WHERE ticker = '{TICKER}';",
    'stock_pnl': f"SELECT * FROM stock_pnl WHERE ticker = '{TICKER}';",
    'stock_balancesheet': f"SELECT * FROM stock_balancesheet WHERE ticker = '{TICKER}';",
    'stock_cashflow': f"SELECT * FROM stock_cashflow WHERE ticker = '{TICKER}';",
    'stock_analyst_reco': f"SELECT * FROM stock_analyst_reco WHERE ticker = '{TICKER}';",
    'stock_valuation': f"SELECT * FROM stock_valuation WHERE ticker = '{TICKER}';",
    'nse_price_history': f"SELECT * FROM nse_price_history WHERE nse_ticker = '{TICKER}';"
}

# Helper function to convert Decimal to float
def convert_decimals_to_float(df):
    """
    Convert all decimal.Decimal types in the DataFrame to float.
    """
    for col in df.columns:
        if df[col].dtype == 'object' or isinstance(df[col].dtype, decimal.Decimal):
            try:
                df[col] = df[col].apply(lambda x: float(x) if isinstance(x, decimal.Decimal) else x)
            except:
                pass
    return df

# Database Connection and Data Fetching
def fetch_data_from_db():
    connection = pymysql.connect(
        host=DB_CONFIG['host'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password'],
        database=DB_CONFIG['database']
    )
    
    data = {}
    try:
        with connection.cursor() as cursor:
            for key, query in QUERIES.items():
                cursor.execute(query)
                result = cursor.fetchall()
                columns = [desc[0] for desc in cursor.description]
                df = pd.DataFrame(result, columns=columns)
                data[key] = convert_decimals_to_float(df)  # Convert decimals to float
    finally:
        connection.close()
    
    return data

# EMS1-EMS7: Profit & Loss Shenanigans
def check_profit_loss_shenanigans(pnl_data):
    print("\n### Profit & Loss Shenanigans ###")
    
    pnl_data['Revenue Growth (%)'] = pnl_data['sales'].astype(float).pct_change() * 100
    pnl_data['Profit Growth (%)'] = pnl_data['net_profit'].astype(float).pct_change() * 100
    pnl_data['OPM (%)'] = pnl_data['operating_profit'].astype(float) / pnl_data['sales'].astype(float) * 100
    
    results = {}
    
    # EMS1: Unusual growth in revenue or profit
    unusual_growth = pnl_data[(pnl_data['Revenue Growth (%)'] > 50) | (pnl_data['Profit Growth (%)'] > 50)]
    results['Unusual Growth'] = unusual_growth if not unusual_growth.empty else "No unusual growth detected."

    # EMS2: Margin analysis
    opm_volatility = pnl_data['OPM (%)'].std()
    results['OPM Volatility'] = f"Standard deviation of OPM: {opm_volatility:.2f}"

    return results

# CFS1-CFS3: Cash Flow Shenanigans
def check_cash_flow_shenanigans(cf_data):
    print("\n### Cash Flow Shenanigans ###")
    
    results = {}
    
    # CFS1: Negative or declining operating cash flow
    cf_data['CFO Decline'] = cf_data['cash_from_operating_activity'].astype(float).diff() < 0
    negative_cfo = cf_data[cf_data['cash_from_operating_activity'].astype(float) < 0]
    results['Negative CFO'] = negative_cfo if not negative_cfo.empty else "No negative CFO detected."

    # CFS2: Divergence between net income and CFO
    cf_data['CFO/Net Income'] = cf_data['cash_from_operating_activity'].astype(float) / cf_data['net_cashflow'].astype(float)
    cfo_divergence = cf_data[cf_data['CFO/Net Income'] < 0.8]
    results['CFO Divergence'] = cfo_divergence if not cfo_divergence.empty else "No significant CFO divergence detected."

    return results

# KMS1-KMS2: Key Metrics Shenanigans
def check_key_metrics_shenanigans(bs_data):
    print("\n### Key Metrics Shenanigans ###")
    
    # Ensure all columns are converted to lowercase for consistent access
    bs_data.columns = bs_data.columns.str.lower()
    
    # Handle missing or derived columns
    if 'current_assets' not in bs_data.columns:
        if {'debtors', 'inventory'}.issubset(bs_data.columns):
            bs_data['current_assets'] = bs_data['debtors'].astype(float) + bs_data['inventory'].astype(float)
        else:
            raise KeyError("Missing required columns to compute 'current_assets'. Please check the table schema.")
    
    if 'current_liabilities' not in bs_data.columns:
        if 'other_liabilities' in bs_data.columns:
            bs_data['current_liabilities'] = bs_data['other_liabilities'].astype(float)
        else:
            raise KeyError("Missing required columns to compute 'current_liabilities'. Please check the table schema.")

    results = {}
    
    # KMS1: Increasing leverage
    if 'borrowings' in bs_data.columns and 'reserves' in bs_data.columns:
        bs_data['debt_to_equity'] = bs_data['borrowings'].astype(float) / bs_data['reserves'].astype(float)
        high_leverage = bs_data[bs_data['debt_to_equity'] > 2]
        results['High Leverage'] = high_leverage if not high_leverage.empty else "No high leverage detected."
    else:
        results['High Leverage'] = "Insufficient data to calculate Debt-to-Equity ratio."

    # KMS2: Declining working capital
    bs_data['working_capital'] = bs_data['current_assets'] - bs_data['current_liabilities']
    declining_wc = bs_data[bs_data['working_capital'].diff() < 0]
    results['Declining Working Capital'] = declining_wc if not declining_wc.empty else "No decline in working capital detected."

    return results

# Consolidate results and generate report
def generate_report(results):
    report = "\n### Financial Shenanigans Report ###\n"
    for category, findings in results.items():
        report += f"\n{category}:\n"
        if isinstance(findings, pd.DataFrame):
            report += findings.to_string(index=False) + "\n"
        else:
            report += f"{findings}\n"
    return report

# Main function to run all checks
def run_financial_shenanigans_analysis():
    data = fetch_data_from_db()
    
    # Perform checks
    pnl_results = check_profit_loss_shenanigans(data['stock_pnl'])
    cf_results = check_cash_flow_shenanigans(data['stock_cashflow'])
    bs_results = check_key_metrics_shenanigans(data['stock_balancesheet'])
    
    # Consolidate results
    all_results = {
        "Profit & Loss Shenanigans": pnl_results,
        "Cash Flow Shenanigans": cf_results,
        "Key Metrics Shenanigans": bs_results
    }
    
    # Generate report
    report = generate_report(all_results)
    print(report)

# Run the analysis
run_financial_shenanigans_analysis()


### Profit & Loss Shenanigans ###

### Cash Flow Shenanigans ###

### Key Metrics Shenanigans ###

### Financial Shenanigans Report ###

Profit & Loss Shenanigans:
{'Unusual Growth':        ticker cons_standalone        date   sales  expenses  operating_profit  \
2  VENUSPIPES               C  2021-03-31  309.33    274.55             34.78   
5  VENUSPIPES               C  2024-03-31  802.20    655.89            146.31   

   other_income  depreciation  interest  profit_before_tax  ...  avg_roic_5yr  \
2          2.70          0.97      5.56              30.95  ...        0.0000   
5          3.18         11.77     22.08             115.64  ...        0.1551   

   avg_roic_10yr  avg_roic  roic_min  roic_max  avg_ppe_to_sales  \
2            0.0    0.0000    0.0000    0.0000            0.0000   
5            0.0    0.1551    0.1507    0.1562            0.1782   

   avg_dep_to_ppe  Revenue Growth (%)  Profit Growth (%)    OPM (%)  
2          0.0000           73.966594         472.154